## AlphaEarth Interactive Mapping — Draw AOI → Real-time Change Detection

**Purpose**: Interactive UI for change detection on a user-defined area of interest (AOI).
Draw a polygon/rectangle → compute AlphaEarth dissimilarity → inspect point-level timeseries.

**Inputs**: `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` (2017–2024)
**Outputs**: Optional export to GEE Asset or Google Drive (requires GEE write access)
**GEE auth required**: Yes

**UI Controls**:
- Draw tool (toolbar): sketch AOI polygon or rectangle
- Magnitude threshold slider (default 0.15)
- Gaussian smoothing radius / sigma sliders
- Layer opacity sliders
- Inspector: click any map point to see year-by-year similarity chart
- Export button: write results to GEE Asset or Drive

**Expected runtime**: < 1 min to load UI; AOI computation ~30 s server-side

**How to run**: Run all cells, then interact with the displayed map widget.


## Interactive Alpha Earth Change Detection

In [ ]:
# ── Repo root on sys.path (works from repo root or notebooks/) ──────────────────
import sys
from pathlib import Path
_repo_root = Path.cwd()
if not (_repo_root / "src").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# Setup: imports and Earth Engine init
import ee, geemap
from ipyleaflet import DrawControl, Marker, LayersControl
import ipywidgets as widgets
from ipywidgets import HTML, VBox, HBox, Label
import math
from datetime import datetime

from src.config import GEE_PROJECT

print("Authenticating/initializing Earth Engine…")
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    print("No active EE session found. Launching authentication flow…")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

# Basic instructions
print("Draw a polygon/rectangle, set params, then click 'Compute change'.")

In [ ]:
# ── Run-mode parameters ─────────────────────────────────────────────────────────
from src.config import AE_MAG_THRESHOLD, AE_SMOOTH_RADIUS, AE_SMOOTH_SIGMA

# INTERACTIVE_MODE = True  → show draw UI; user draws AOI, then clicks Compute
# INTERACTIVE_MODE = False → headless: auto-runs on DEMO_AOI (CI / reproducibility)
INTERACTIVE_MODE = True

# Demo AOI used when INTERACTIVE_MODE = False (Austin, TX area)
DEMO_AOI = ee.Geometry.Rectangle([-98.0, 30.1, -97.6, 30.5])

In [ ]:
# Visualization parameters — loaded from shared module
from src.visualization import VIS_YOD, VIS_MAG_AE, VIS_MAG_LT, VIS_DUR

# Local aliases matching the names used throughout this notebook
vis_yod   = VIS_YOD
vis_mag_a = VIS_MAG_AE
vis_mag_l = VIS_MAG_LT
vis_dur_a = VIS_DUR
vis_dur_l = VIS_DUR

In [ ]:
# Helpers: AlphaEarth change layers — imported from shared module
from src.config import ALPHAEARTH_COLLECTION, AE_YEAR_START, AE_YEAR_END
from src.change_detection import compute_alpha_layers, smooth_and_mask_alpha, normalize_image

# Keep embeddings + years in module scope: used by the Inspector cell below
embeddings = ee.ImageCollection(ALPHAEARTH_COLLECTION)
years = list(range(AE_YEAR_START, AE_YEAR_END + 1))

# Alias so the Inspector callback (_on_map_click2) works without modification
_normalize = normalize_image

In [ ]:
# Interactive map with draw + compute UI
Map = geemap.Map(center=[37.5, -98], zoom=4)
Map.add_basemap('Esri.WorldImagery')
Map.add_layer_control()

# Remove default geemap draw control to avoid duplicate icons
try:
    if hasattr(Map, 'draw_control') and Map.draw_control is not None:
        Map.remove_control(Map.draw_control)
except Exception:
    pass

draw = DrawControl(
    polyline={},
    circlemarker={},
    rectangle={"shapeOptions": {"color": "#00ffff", "opacity": 1.0, "weight": 2, "fillColor": "#00ffff", "fillOpacity": 0.0}},
    polygon={"shapeOptions": {"color": "#00ffff", "opacity": 1.0, "weight": 2, "fillColor": "#00ffff", "fillOpacity": 0.0}},
    circle={}
 )
Map.add_control(draw)

# AOI state
aoi_geom = {'ee': None}

status = widgets.HTML(value="<b>Step 1:</b> Draw a polygon/rectangle AOI on the map.")
btn_clear = widgets.Button(description='Clear AOI', button_style='warning')
btn_run = widgets.Button(description='Compute change', button_style='primary')

# Parameters
mag_thresh = widgets.FloatSlider(description='AE MAG thr', value=0.15, min=0.05, max=0.5, step=0.01, readout_format='.2f')
smooth_radius = widgets.IntSlider(description='Smooth radius', value=2, min=0, max=5)
smooth_sigma = widgets.FloatSlider(description='Smooth sigma', value=1.0, min=0.25, max=3, step=0.25)
opacity = widgets.FloatSlider(description='Layer opacity', value=0.85, min=0.2, max=1.0, step=0.05)

ui = VBox([status, HBox([btn_clear, btn_run]), HBox([mag_thresh, smooth_radius, smooth_sigma, opacity])])

# Capture draw events
def _to_ee_geometry(feature):
    geom_type = feature['geometry']['type']
    coords = feature['geometry']['coordinates']
    if geom_type == 'Polygon':
        return ee.Geometry.Polygon(coords)
    if geom_type == 'Rectangle':  # ipyleaflet sends Polygon for rect; kept for future
        return ee.Geometry.Polygon(coords)
    if geom_type == 'LineString':
        return ee.Geometry.LineString(coords)
    if geom_type == 'Point':
        return ee.Geometry.Point(coords)
    # default try polygon
    return ee.Geometry.Polygon(coords)

def _on_draw(target, action=None, geo_json=None):
    if action in ('created', 'edited') and geo_json:
        try:
            geom = _to_ee_geometry(geo_json)
            aoi_geom['ee'] = geom
            status.value = "<b>AOI set.</b> Click 'Compute change' to run AlphaEarth method."
        except Exception as e:
            status.value = f"Error parsing geometry: {e}"
    elif action == 'deleted':
        aoi_geom['ee'] = None
        status.value = "AOI cleared. Draw again."

draw.on_draw(_on_draw)

def _clear_layers(prefixes=('AE: ',)):
    # Remove previous layers matching prefixes to keep map tidy
    to_remove = []
    for lyr in list(Map.layers):
        name = getattr(lyr, 'name', '') or ''
        if any(name.startswith(p) for p in prefixes):
            to_remove.append(lyr)
    for lyr in to_remove:
        try:
            Map.remove_layer(lyr)
        except Exception:
            pass

def _run_compute(_):
    if aoi_geom['ee'] is None:
        status.value = "<span style='color:red'>Please draw an AOI first.</span>"
        return
    status.value = "Running computation on server (AlphaEarth)… please wait…"
    geom = aoi_geom['ee']

    # Remove previously added AE layers for clean re-run
    _clear_layers()

    # Compute raw AlphaEarth layers
    yoc, mag, dur = compute_alpha_layers(geom, change_threshold=mag_thresh.value)
    # Add raw layers (clear names, consistent order)
    Map.addLayer(yoc, vis_yod, 'AE: Raw - YOC', False, opacity.value)
    Map.addLayer(mag, vis_mag_a, 'AE: Raw - MAG', False, opacity.value)
    Map.addLayer(dur, vis_dur_a, 'AE: Raw - DUR', False, opacity.value)

    # Smooth + mask
    mag_s, chg_mask, yoc_m, mag_m, dur_m = smooth_and_mask_alpha(
        yoc, mag, dur, mag_threshold=mag_thresh.value, radius=smooth_radius.value, sigma=smooth_sigma.value
    )
    # Add masked/smoothed layers (decision-ready)
    Map.addLayer(yoc_m, vis_yod, 'AE: Masked - YOC', False, opacity.value)
    Map.addLayer(mag_m, vis_mag_a, 'AE: Masked - MAG', True, opacity.value)
    Map.addLayer(dur_m, vis_dur_a, 'AE: Masked - DUR', False, opacity.value)
    Map.addLayer(mag_s, vis_mag_a, 'AE: Smoothed - MAG', False, opacity.value)
    Map.addLayer(chg_mask.selfMask(), {'min':0, 'max':1, 'palette':['#00ff00']}, 'AE: Change Mask', False, 0.8)

    # Outline AOI for reference (no fill)
    Map.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(geom)]), 1, 2),
                  {'palette': ['#00ffff']}, 'AE: AOI Outline', True)

    status.value = "Done. Toggle raw vs masked layers in the control to compare."

btn_run.on_click(_run_compute)
btn_clear.on_click(lambda _:(
    draw.clear(),
    _clear_layers(),
    aoi_geom.update({'ee': None}),
    setattr(status, 'value', 'AOI cleared. Draw again.')
))

display(ui)
Map

In [ ]:
# ── Headless / non-interactive auto-run ─────────────────────────────────────────
if not INTERACTIVE_MODE:
    print("INTERACTIVE_MODE=False: running AlphaEarth on DEMO_AOI…")
    aoi_geom['ee'] = DEMO_AOI
    mag_thresh.value      = AE_MAG_THRESHOLD
    smooth_radius.value   = AE_SMOOTH_RADIUS
    smooth_sigma.value    = AE_SMOOTH_SIGMA
    _run_compute(None)
    print("Done. Layers added to Map (display Map above to inspect).")


## Inspector + Export (AlphaEarth Change)
Click inside the AOI to see year-to-year cosine similarity at a point. Use the Export panel to send layers to a GEE Asset folder or Google Drive.

In [ ]:
# New interactive map: Inspector + Export
import matplotlib.pyplot as plt
from ipywidgets import Output
from IPython.display import display, clear_output
from ipyleaflet import Popup
import io, base64

# Reuse vis params, helpers, and embeddings/years from above
Map2 = geemap.Map(center=[37.5, -98], zoom=4)
Map2.add_basemap('Esri.WorldImagery')
Map2.add_layer_control()

# Remove default geemap draw control to avoid duplicate icons
try:
    if hasattr(Map2, 'draw_control') and Map2.draw_control is not None:
        Map2.remove_control(Map2.draw_control)
except Exception:
    pass

draw2 = DrawControl(
    polyline={}, circlemarker={},
    rectangle={"shapeOptions": {"color": "#ffa500", "opacity": 1.0, "weight": 2, "fillOpacity": 0.0}},
    polygon={"shapeOptions": {"color": "#ffa500", "opacity": 1.0, "weight": 2, "fillOpacity": 0.0}},
    circle={}
 )
Map2.add_control(draw2)

# State
aoi2 = {'ee': None}
status2 = widgets.HTML(value="<b>Inspector Map:</b> Draw AOI, Compute, then Click within AOI to inspect.")
btn_clear2 = widgets.Button(description='Clear AOI', button_style='warning')
btn_run2 = widgets.Button(description='Compute change', button_style='primary')

# Inspector + export UI
out_plot = Output()  # kept as secondary visualization
btn_export = widgets.Button(description='Export…', button_style='')
export_dest = widgets.ToggleButtons(options=['GEE Asset','Google Drive'], value='Google Drive', description='To:')
export_name = widgets.Text(value='alphaearth_change', description='Name:')
asset_folder = widgets.Text(value='users/your_username', description='Asset/Fldr:')
export_scale = widgets.IntSlider(value=30, min=10, max=200, step=10, description='Scale (m):')

mag_thresh2 = widgets.FloatSlider(description='AE MAG thr', value=0.15, min=0.05, max=0.5, step=0.01, readout_format='.2f')
smooth_radius2 = widgets.IntSlider(description='Smooth radius', value=2, min=0, max=5)
smooth_sigma2 = widgets.FloatSlider(description='Smooth sigma', value=1.0, min=0.25, max=3, step=0.25)
opacity2 = widgets.FloatSlider(description='Layer opacity', value=0.85, min=0.2, max=1.0, step=0.05)

ui2_top = VBox([status2, HBox([btn_clear2, btn_run2]), HBox([mag_thresh2, smooth_radius2, smooth_sigma2, opacity2])])
ui2_export = VBox([HBox([export_dest, export_name]), HBox([asset_folder, export_scale]), btn_export])

# Conversion and draw handler
def _to_ee(feature):
    t = feature['geometry']['type']
    coords = feature['geometry']['coordinates']
    if t == 'Polygon':
        return ee.Geometry.Polygon(coords)
    if t == 'Rectangle':
        return ee.Geometry.Polygon(coords)
    if t == 'LineString':
        return ee.Geometry.LineString(coords)
    if t == 'Point':
        return ee.Geometry.Point(coords)
    return ee.Geometry.Polygon(coords)

def _on_draw2(target, action=None, geo_json=None):
    if action in ('created','edited') and geo_json:
        try:
            aoi2['ee'] = _to_ee(geo_json)
            status2.value = "<b>AOI set.</b> Click Compute change, then click on the map to inspect."
        except Exception as e:
            status2.value = f"Error: {e}"
    elif action == 'deleted':
        aoi2['ee'] = None
        status2.value = "AOI cleared. Draw again."

draw2.on_draw(_on_draw2)

# Layers state for export
layers2 = {'raw': None, 'masked': None, 'smoothed': None, 'mask': None, 'aoi_outline': None}

# Helper to clear AE2 layers/popups and AOI drawings
def _clear2(_=None):
    # Clear drawn features
    try:
        draw2.clear()
    except Exception:
        pass
    # Remove AE2 layers
    to_remove = []
    for lyr in list(Map2.layers):
        nm = getattr(lyr, 'name', '') or ''
        if nm.startswith('AE2: '):
            to_remove.append(lyr)
    for lyr in to_remove:
        try:
            Map2.remove_layer(lyr)
        except Exception:
            pass
    # Reset state and status
    aoi2['ee'] = None
    status2.value = "AOI cleared. Draw again."
    # Remove popup if any
    if _last_popup2.get('p') is not None:
        try:
            Map2.remove(_last_popup2['p'])
        except Exception:
            pass
        _last_popup2['p'] = None
    # Clear side plot
    with out_plot:
        clear_output(wait=True)

btn_clear2.on_click(_clear2)


def _compute2(_):
    if aoi2['ee'] is None:
        status2.value = "<span style='color:red'>Please draw an AOI first.</span>"
        return
    geom = aoi2['ee']
    status2.value = "Computing AlphaEarth layers…"
    # Compute
    yoc, mag, dur = compute_alpha_layers(geom, change_threshold=mag_thresh2.value)
    mag_s, chg_mask, yoc_m, mag_m, dur_m = smooth_and_mask_alpha(
        yoc, mag, dur, mag_threshold=mag_thresh2.value, radius=smooth_radius2.value, sigma=smooth_sigma2.value)
    # Clear prior AE2 layers
    for lyr in list(Map2.layers):
        nm = getattr(lyr, 'name', '') or ''
        if nm.startswith('AE2: '):
            try: Map2.remove_layer(lyr)
            except Exception: pass
    # Add layers with AE2: prefix
    Map2.addLayer(yoc, vis_yod, 'AE2: Raw - YOC', False, opacity2.value)
    Map2.addLayer(mag, vis_mag_a, 'AE2: Raw - MAG', False, opacity2.value)
    Map2.addLayer(dur, vis_dur_a, 'AE2: Raw - DUR', False, opacity2.value)
    Map2.addLayer(yoc_m, vis_yod, 'AE2: Masked - YOC', False, opacity2.value)
    Map2.addLayer(mag_m, vis_mag_a, 'AE2: Masked - MAG', True, opacity2.value)
    Map2.addLayer(dur_m, vis_dur_a, 'AE2: Masked - DUR', False, opacity2.value)
    Map2.addLayer(mag_s, vis_mag_a, 'AE2: Smoothed - MAG', False, opacity2.value)
    Map2.addLayer(chg_mask.selfMask(), {'min':0,'max':1,'palette':['#00ff00']}, 'AE2: Change Mask', False, 0.8)
    Map2.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(geom)]), 1, 2), {'palette':['#ffa500']}, 'AE2: AOI Outline', True)
    # Save for export
    layers2['raw'] = {'yoc': yoc, 'mag': mag, 'dur': dur}
    layers2['masked'] = {'yoc_m': yoc_m, 'mag_m': mag_m, 'dur_m': dur_m}
    layers2['smoothed'] = {'mag_s': mag_s}
    layers2['mask'] = {'mask': chg_mask}
    layers2['aoi_outline'] = {'aoi': geom}
    status2.value = "Done. Click a point inside AOI to inspect. Use Export panel to save layers."

btn_run2.on_click(_compute2)

# Inspector: click to plot pairwise cosine DISsimilarity at a point, shown as popup
_last_popup2 = {'p': None}

def _plot_png(yrs, vals, title='1 - Cosine similarity by year'):
    fig, ax = plt.subplots(figsize=(4.2, 2.6), dpi=150)
    ax.plot(yrs, vals, marker='o', color='#d53e4f')
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Year (end of pair)')
    ax.set_ylabel('1 - cosine')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    return f"<img src='data:image/png;base64,{b64}' width='320'/>"

def _on_map_click2(**kwargs):
    if kwargs.get('type') != 'click':
        return
    if aoi2['ee'] is None:
        status2.value = "<span style='color:red'>Please set AOI first.</span>"
        return
    lat, lon = kwargs.get('coordinates')
    pt = ee.Geometry.Point([lon, lat])
    # Ensure click is inside AOI (use contains)
    try:
        inside = aoi2['ee'].contains(pt, 1).getInfo()
    except Exception:
        inside = True  # fallback
    if not inside:
        status2.value = "<span style='color:red'>Click is outside AOI.</span>"
        return
    # Build per-year normalized images and compute cosines
    cos_vals = []
    yrs = []
    for i in range(len(years)-1):
        yA = years[i]
        yB = years[i+1]
        imgA = (embeddings
                 .filterDate(ee.Date.fromYMD(yA,1,1), ee.Date.fromYMD(yA+1,1,1))
                 .filterBounds(aoi2['ee'])
                 .mosaic())
        imgB = (embeddings
                 .filterDate(ee.Date.fromYMD(yB,1,1), ee.Date.fromYMD(yB+1,1,1))
                 .filterBounds(aoi2['ee'])
                 .mosaic())
        A = _normalize(imgA)
        B = _normalize(imgB)
        cos = A.multiply(B).reduce(ee.Reducer.sum())  # cosine similarity
        try:
            val = cos.sample(pt, 30).first().get('sum').getInfo()
        except Exception:
            val = None
        if val is None:
            continue
        cos_vals.append(val)
        yrs.append(yB)
    if not cos_vals:
        status2.value = "<span style='color:red'>No data at this point.</span>"
        return
    # Dissimilarity = 1 - cosine
    dissim = [max(0.0, min(1.0, 1.0 - v)) for v in cos_vals]
    html_img = _plot_png(yrs, dissim, title='Year-to-year dissimilarity (1 - cosine)')
    # Remove previous popup
    if _last_popup2['p'] is not None:
        try:
            Map2.remove(_last_popup2['p'])
        except Exception:
            pass
        _last_popup2['p'] = None
    # Show popup at click location
    popup = Popup(location=(lat, lon), child=HTML(value=html_img), close_button=True, auto_close=True, close_on_escape_key=True)
    Map2.add(popup)
    _last_popup2['p'] = popup
    # Also reflect in side output (optional)
    with out_plot:
        clear_output(wait=True)
        display(HTML(value=html_img))

Map2.on_interaction(_on_map_click2)

# Export handlers
def _export_click(_):
    if aoi2['ee'] is None:
        status2.value = "<span style='color:red'>Please set AOI first.</span>"
        return
    geom = aoi2['ee']
    dest = export_dest.value
    name = export_name.value.strip() or 'alphaearth_change'
    scale = int(export_scale.value)
    region = geom.bounds(1)
    # Prepare dict of label->image
    to_export = {
        'AE2_Raw_YOC': layers2['raw']['yoc'],
        'AE2_Raw_MAG': layers2['raw']['mag'],
        'AE2_Raw_DUR': layers2['raw']['dur'],
        'AE2_Masked_YOC': layers2['masked']['yoc_m'],
        'AE2_Masked_MAG': layers2['masked']['mag_m'],
        'AE2_Masked_DUR': layers2['masked']['dur_m'],
        'AE2_Smoothed_MAG': layers2['smoothed']['mag_s'],
        'AE2_Change_Mask': layers2['mask']['mask'].toInt8()
    }
    # Launch tasks
    launched = []
    for label, img in to_export.items():
        task_name = f"{name}_{label}"
        if dest == 'GEE Asset':
            # Destination path: asset_folder/name_label
            asset_id = f"{asset_folder.value.rstrip('/')}/{task_name}"
            task = ee.batch.Export.image.toAsset(image=img, description=task_name, assetId=asset_id, scale=scale, region=region, maxPixels=1e13)
        else:
            task = ee.batch.Export.image.toDrive(image=img, description=task_name, folder=asset_folder.value or None, fileNamePrefix=task_name, scale=scale, region=region, maxPixels=1e13)
        task.start()
        launched.append(task_name)
    status2.value = "Export started for: " + ', '.join(launched)

btn_export.on_click(_export_click)

display(ui2_top)
display(ui2_export)
display(out_plot)
Map2